# 🎨 04 · 图文生成与扩散模型

> 文本→图像的主力是**扩散模型（Diffusion）**：先正向加噪，再学习反向去噪。
> 本节手写**加噪/去噪/采样**全流程，并讲清**文本条件注入**（Classifier-Free Guidance）思想。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 扩散模型的思想？ | 前向加噪 → 反向去噪 |
| 2 | 怎么训练？ | 预测噪声 ε |
| 3 | 怎么采样？ | 迭代去噪（DDPM） |
| 4 | 文本怎么控制生成？ | 条件注入 + CFG |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 扩散模型总览（必背）

In [2]:
print('''扩散模型（如 Stable Diffusion）：
  前向：对图像 x0 逐步加高斯噪声，T 步后变成纯噪声 xT；
  反向：学一个网络 ε_θ(x_t, t, 文本) 预测噪声，从 xT 逐步去噪回到 x0；
  训练目标：最小化 预测噪声 与 真实噪声 的 MSE；
  采样：随机噪声 -> 去噪 T 步 -> 生成图像。

常见家族：DDPM（基础）、DDIM（加速采样）、LDM/SD（在潜空间扩散）、
        DiT（用 Transformer 替代 UNet，SD3/Flux 的路子）。
''')

扩散模型（如 Stable Diffusion）：
  前向：对图像 x0 逐步加高斯噪声，T 步后变成纯噪声 xT；
  反向：学一个网络 ε_θ(x_t, t, 文本) 预测噪声，从 xT 逐步去噪回到 x0；
  训练目标：最小化 预测噪声 与 真实噪声 的 MSE；
  采样：随机噪声 -> 去噪 T 步 -> 生成图像。

常见家族：DDPM（基础）、DDIM（加速采样）、LDM/SD（在潜空间扩散）、
        DiT（用 Transformer 替代 UNet，SD3/Flux 的路子）。



## 2️⃣ 前向加噪（手写）

In [3]:
# 简化 1D 信号当作"图像"
def q_sample(x0, t, alpha_bar):
    # x_t = sqrt(alpha_bar_t) * x0 + sqrt(1-alpha_bar_t) * eps
    eps = rng.normal(0, 1, x0.shape)
    return np.sqrt(alpha_bar[t]) * x0 + np.sqrt(1 - alpha_bar[t]) * eps, eps

T = 50
alpha_bar = np.linspace(1.0, 0.02, T)     # 噪声调度：前几步几乎无损，后几步全噪声
x0 = np.zeros(64); x0[20:44] = 1.0        # 简单"图像"：亮条
xt, eps = q_sample(x0, 30, alpha_bar)
print(f'x0 均值={x0.mean():.3f}，x_t(t=30) 均值={xt.mean():.3f}')
print('✅ 前向加噪完成（信号逐渐被噪声淹没）')

x0 均值=0.375，x_t(t=30) 均值=0.269
✅ 前向加噪完成（信号逐渐被噪声淹没）


## 3️⃣ 训练：预测噪声（简化线性拟合演示）

In [4]:
# 用线性回归学习"从 x_t,t 预测噪声"（极度简化，示意训练目标）
t_idx = rng.integers(0, T, 2000)
x0_data = rng.normal(0, 1, (2000, 64))
eps_data = rng.normal(0, 1, (2000, 64))
xt_data = np.sqrt(alpha_bar[t_idx, None]) * x0_data + np.sqrt(1 - alpha_bar[t_idx, None]) * eps_data

from sklearn.linear_model import LinearRegression
feat = xt_data
model = LinearRegression().fit(feat, eps_data)
pred_eps = model.predict(feat)
mse = np.mean((pred_eps - eps_data) ** 2)
print(f'噪声预测 MSE（线性基线）: {mse:.4f}')
print('➡️ 真实场景用 UNet/DiT 做条件网络，MSE 训练目标相同；线性基线只是演示训练范式')

噪声预测 MSE（线性基线）: 0.5531
➡️ 真实场景用 UNet/DiT 做条件网络，MSE 训练目标相同；线性基线只是演示训练范式


## 4️⃣ 反向采样：去噪生成（手写 DDPM 采样）

In [5]:
# 简化采样：已知"干净信号"是 0/1 亮条，直接从噪声逐步"去噪"（演示迭代原理）
x = rng.normal(0, 1, 64)
for t in reversed(range(T)):
    alpha = alpha_bar[t]
    # 简化去噪步：按当前噪声水平向"预测均值"靠近（真实 DDPM 用网络输出）
    denoised = np.clip(x / np.sqrt(alpha + 1e-6), 0, 1) if alpha > 0.1 else x
    x = 0.98 * denoised + 0.02 * rng.normal(0, 1, 64)
recovered = x

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1); plt.bar(range(64), x0, width=1); plt.title('原始信号 x0')
plt.subplot(1, 2, 2); plt.bar(range(64), recovered, width=1, color='#10b981'); plt.title('采样恢复（示意）')
plt.tight_layout()
plt.show()
print(f'恢复信号与原始的相关性: {np.corrcoef(x0, recovered)[0,1]:.3f}')

恢复信号与原始的相关性: 0.035


C:\Users\24260\AppData\Local\Temp\ipykernel_46276\1251864585.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5️⃣ 文本条件与 Classifier-Free Guidance（CFG）

In [6]:
print('''文本条件注入方式（SD 的做法）：
  1) 文本 -> CLIP 文本编码器 -> 条件向量 c
  2) 在 UNet 的注意力/跨注意力层注入 c（cross-attention）
  3) 训练时随机丢弃条件（10%）让模型学会"无条件生成"

CFG（Classifier-Free Guidance）采样：
  eps_guided = eps_uncond + w * (eps_cond - eps_uncond)
  w>1 增强文本跟随度；w=7 是常见值（SD 默认）。
''')
# 演示 CFG 对"方向强度"的影响：w 越大越贴合条件
w_values = [0, 3, 7]
for w in w_values:
    # 简单示意：有条件方向偏移放大
    direction = rng.normal(0, 1, 8)
    eps_un = rng.normal(0, 1, 8)
    guided = eps_un + w * direction - w * eps_un * 0  # 示意公式形态
    print(f'w={w}: CFG 引导向量范数={np.linalg.norm(guided):.2f}（w 越大越贴近条件）')

文本条件注入方式（SD 的做法）：
  1) 文本 -> CLIP 文本编码器 -> 条件向量 c
  2) 在 UNet 的注意力/跨注意力层注入 c（cross-attention）
  3) 训练时随机丢弃条件（10%）让模型学会"无条件生成"

CFG（Classifier-Free Guidance）采样：
  eps_guided = eps_uncond + w * (eps_cond - eps_uncond)
  w>1 增强文本跟随度；w=7 是常见值（SD 默认）。

w=0: CFG 引导向量范数=1.34（w 越大越贴近条件）
w=3: CFG 引导向量范数=11.34（w 越大越贴近条件）
w=7: CFG 引导向量范数=20.73（w 越大越贴近条件）


## 6️⃣ 生成模型家族（面试速答）

In [7]:
print('''| 家族 | 代表 | 特点 |
|---|---|---|
| VAE | VAE/VQ-VAE | 学习隐空间分布 |
| GAN | StyleGAN | 快但训练不稳 |
| 自回归 | DALL·E 1 | 图像离散化后逐 token 生 |
| 扩散 | DDPM/DDIM/SD/DiT | 质量高、可控、当前主流 |
| 混合 | SD + ControlNet/LoRA | 可控生成/个性化 |
''')

| 家族 | 代表 | 特点 |
|---|---|---|
| VAE | VAE/VQ-VAE | 学习隐空间分布 |
| GAN | StyleGAN | 快但训练不稳 |
| 自回归 | DALL·E 1 | 图像离散化后逐 token 生 |
| 扩散 | DDPM/DDIM/SD/DiT | 质量高、可控、当前主流 |
| 混合 | SD + ControlNet/LoRA | 可控生成/个性化 |



## 📝 自测清单

- [ ] 能手写前向加噪（q_sample）并解释噪声调度
- [ ] 能说出扩散训练目标（预测噪声 MSE）
- [ ] 能手写/口述反向采样迭代流程
- [ ] 能解释 CFG 采样公式与 w 的作用
- [ ] 能背出生成模型家族对比